In [17]:
import os, sys
import pandas as pd
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))
from src.semantic import semantic_retriever_rag, load_semantic
from src.rag_pipeline import _rag_answer
from src.utils import clean_output

In [18]:
load_dotenv()
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [19]:


llm_endpoint_llama = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation",
    max_new_tokens=200,
    huggingfacehub_api_token=hf_token,
)

llm_llama_M2 = ChatHuggingFace(llm=llm_endpoint_llama)




In [20]:
response_llama = llm_llama_M2.invoke("Say hello in one sentence.")
print("Llama:", response_llama.content)


Llama: Hello, how are you today?


In [21]:
llm_endpoint_qwen = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-8B",
    task="text-generation",
    max_new_tokens=200,
    huggingfacehub_api_token=hf_token
)

llm_qwen_new = ChatHuggingFace(llm=llm_endpoint_qwen)

In [22]:
response_llama = llm_qwen_new.invoke("Say hello in one sentence.")
print("Llama:", response_llama.content)

Llama: <think>
Okay, the user asked me to say hello in one sentence. Let me think about how to respond.

First, I need to make sure the response is friendly and concise. Since it's just a simple greeting, maybe something like "Hello! How can I assist you today?" That's straightforward and opens the door for further interaction. 

Wait, should I add more personality? Maybe a smiley emoji to keep it light. But the user didn't specify any preferences about emojis. Maybe stick to the basics. 

Also, the user might be testing if I can follow simple instructions. Keeping it simple and polite is key. Let me check for any typos. "Hello! How can I assist you today?" seems good. 

I think that's it. It's a standard greeting that's appropriate for most situations. No need to overcomplicate it. Alright, ready to send.
</think>

Hello! How can I assist you today?


In [23]:
merged_df = pd.read_parquet("../data/processed/merged.parquet")
semantic_model, semantic_index, semantic_meta = load_semantic()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11864.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
queries = [
    # From Milestone 1
    "book to relax before bed",
    "books about animals for kids",
     # New ones (for variety)
    "good books for beginners learning investing",
    "books similar to Harry Potter",
    "best mystery books with strong female lead",
]

In [ ]:
comparison_results = []

for q in queries:
    docs = semantic_retriever_rag(
        query=q,
        model=semantic_model,
        index=semantic_index,
        df=merged_df,
        top_k=5
    )

    result_llama = _rag_answer(q, llm_llama_M2, docs)
    result_qwen = _rag_answer(q, llm_qwen_new, docs)

    comparison_results.append({

        "query": q ,
        "docs": docs,
        "llama_answer": result_llama["answer"],
        "qwen_answer": result_qwen["answer"],
        "prompt": result_llama["prompt"]

    })

In [26]:
for r in comparison_results:
    print("\n" + "="*100)
    print(f"Query: {r['query']}")
    
    print("\n--- Llama ---")
    print(r["llama_answer"])
    
    print("\n--- Qwen ---")
    print(clean_output(r["qwen_answer"]))


Query: book to relax before bed

--- Llama ---
The book "The King's Daughter and Other Stories for Girls" (ASIN: B0082VTBB8) put a reviewer's daughter to sleep, and also put a group of adults (including the reviewer's Girl Scout troop) to sleep, suggesting that the book may be a relaxing read. However, the reviewer did not find the book to be inspiring, and mentioned that some stories are "laborious and move too slowly", so the sleep-inducing effect may not be due to the book being interesting enough to keep the reader awake.

--- Qwen ---
<think>
Okay, the user is asking for a book to relax before bed. Let me look through the provided context to find relevant information.

First, I'll check each product review and metadata. The ASIN B09K8X8XC2 has two 5-star reviews. The first review mentions the book is epic and immersive, but it's a dark paranormal gothic romance. The second review also praises the writing as descriptive and thought-provoking, but notes it's dark. Maybe not the bes